In [1]:
import os
from google import genai

# --- Model choice -----------------------------------------------------------
# gemini-3-flash is the free-tier default as of mid-2026 (10 req/min, 1500/day).
# Swap this single string to change models. Nothing else in the notebook
# depends on the specific model.
MODEL_NAME = "gemini-3-flash"

# --- API key ----------------------------------------------------------------
# Read from environment so the key never lives in the notebook / git history.
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not set. In PowerShell run:\n"
        '    $env:GEMINI_API_KEY = "your-key-here"\n'
        "then restart the Jupyter kernel."
    )

client = genai.Client(api_key=API_KEY)

# --- Sources ----------------------------------------------------------------
# mode="http"  -> static HTML, a plain GET works.
# mode="js"    -> JavaScript-rendered, needs a headless browser (out of scope here).


print(f"Model: {MODEL_NAME}")


Model: gemini-3-flash


## 2. The target schema (the "SI units" of this system)

This is the contract. *Every* college, no matter how different its page, must
come out shaped like this. Define the destination format first, before touching
any source — the schema is what makes the sources interchangeable.

Design choices worth noting:

- **`allergens` / `dietary` are normalised lists.** Churchill writes `(VG)`,
  St John's writes the word "Vegan". We push that mess down into the model and
  ask it to emit a controlled vocabulary, so the website never has to know about
  per-college quirks.
- **Everything optional defaults sensibly.** A college that only lists dinner
  should still validate. Brittle schemas that demand every field are a common
  cause of silent data loss.


In [2]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


class Meal(str, Enum):
    """Controlled vocabulary for meal sittings. The model must pick from these."""
    BREAKFAST = "breakfast"
    BRUNCH = "brunch"
    LUNCH = "lunch"
    DINNER = "dinner"


class Dish(BaseModel):
    """A single item on the menu."""
    name: str = Field(description="Dish name, cleaned of allergen codes.")
    course: Optional[str] = Field(
        default=None,
        description="starter | main | side | dessert if the page indicates it, else null.",
    )
    dietary: list[str] = Field(
        default_factory=list,
        description="Normalised tags: any of vegan, vegetarian, halal, gluten-free.",
    )


class MealSitting(BaseModel):
    """All dishes for one meal on one day."""
    meal: Meal
    dishes: list[Dish] = Field(default_factory=list)


class DayMenu(BaseModel):
    """One day's worth of meals."""
    day: str = Field(description="Day of week, e.g. 'Monday'.")
    date: Optional[str] = Field(default=None, description="Date if shown, else null.")
    meals: list[MealSitting] = Field(default_factory=list)


class CollegeMenu(BaseModel):
    """The unified output for one college. This is what the website consumes."""
    college: str
    week_commencing: Optional[str] = Field(default=None)
    days: list[DayMenu] = Field(default_factory=list)


print("Schema defined. The website only ever sees CollegeMenu objects.")

Schema defined. The website only ever sees CollegeMenu objects.


## 3. Stage 1 + 2 — Fetch and Reduce

Two small, separately-testable functions.

`fetch_html` does one job: get the bytes. `html_to_text` strips `<script>`,
`<style>`, `<nav>`, and `<footer>`, then collapses the page to plain text. This
matters for cost: from the recon, the Queens' page is ~80% navigation and footer
boilerplate. Sending that to the model is paying tokens for noise. Reducing first
is the equivalent of filtering sensor noise before it hits your controller.

In [3]:

SOURCES = [
    {"college": "Queens'",     "url": "https://www.queens.cam.ac.uk/life-at-queens/catering/dining-hall/weekly-menu/", "mode": "http"},
    {"college": "Churchill",   "url": "https://www.chu.cam.ac.uk/about/campus/dining-at-college/lunch-and-dinner-menu/", "mode": "http"},
    {"college": "St John's",   "url": "https://menu.joh.cam/",                                                          "mode": "http"},
    {"college": "Wolfson",     "url": "https://www.wolfson.cam.ac.uk/food/cafeteria-menus",                             "mode": "js"},
]

print(f"{len(SOURCES)} sources configured "
      f"({sum(s['mode']=='http' for s in SOURCES)} fetchable now).")


4 sources configured (3 fetchable now).


In [4]:
import requests
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

timeout = 10

url = 'https://menu.joh.cam/'

resp = requests.get(url, headers=HEADERS, timeout=timeout)
resp.raise_for_status()

In [5]:
import requests
from bs4 import BeautifulSoup

# A realistic User-Agent. Some sites return stripped pages to unknown clients.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}


def fetch_html(url: str, timeout: int = 20) -> str:
    """Stage 1: return raw HTML for a URL. Raises on HTTP error."""
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def html_to_text(html: str) -> str:
    """Stage 2: strip boilerplate and return readable plain text.

    We remove script/style/nav/footer/header tags entirely, then extract text
    with newline separators so the day/meal structure survives for the model.
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    # Collapse runs of blank lines so we do not waste tokens on whitespace.
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]
    return "\n".join(lines)


# Quick smoke test on the cleanest source.
_demo = html_to_text(fetch_html(SOURCES[2]["url"]))
print(f"Queens' reduced to {len(_demo)} chars. First 1000:\n")
print(_demo[:1000])

Queens' reduced to 9812 chars. First 1000:

Buttery Menu - St John's College, Cambridge
Friday 19
th
Jun
Lunch
Starter
Roast plum tomato and basil soup
Contains:
Celery
Sulphur Dioxide
Mains
Smokey vegan meatball gumbo with okra, coriander and lemon wedge
Vegan
Contains:
Celery
Gluten (wheat)
Mustard
Soya
Quorn burger, sliced cheese , lettuce and onion confit in a floured bun
Vegetarian
Contains:
Eggs
Milk
Sulphur Dioxide
Beer battered fillet of haddock, tartare sauce and lemon wedge
Contains:
Eggs
Fish
Gluten (wheat, barley)
Sulphur Dioxide
Mustard
Fillet of coley, moqueca prawn stew
Contains:
Celery
Sulphur Dioxide
Crustaceans
Fish
Sides
Chunky chips
Mushy & garden peas
Baked beans
Dessert
Lemon curd and blueberry meringue roulade
Contains:
Eggs
Milk
Cranberry, popcorn and ginger biscuit rocky road
Vegan
Contains:
Gluten (wheat)
Soya
Dinner
Starter
Roast plum tomato and basil soup
Contains:
Celery
Sulphur Dioxide
Mains
Vegan pizza with cherry tomato, chestnut mushrooms and black bean

## 4. Stage 3 — Extract with the LLM

This is the heart of the system. Three things make it robust:

1. **Structured output.** We hand Gemini the Pydantic schema via
   `response_schema` and set `response_mime_type="application/json"`. The model
   is then *constrained* to emit JSON matching our shape — far more reliable than
   asking nicely in the prompt and hoping.
2. **A focused system instruction.** We tell it exactly how to normalise the
   per-college mess (the allergen codes, the dietary words) into our vocabulary.
3. **The reduced text, not raw HTML.** Cheaper and, in practice, just as accurate
   for this kind of page.

The prompt is the same for every college. That is the whole point: one extractor,
N sources.

In [23]:
from google.genai import types

SYSTEM_INSTRUCTION = """\
You extract dining-hall menus from college webpage text into a strict schema.

Rules:
- Only include LUNCH and DINNER style sittings plus brunch/breakfast IF clearly
  present. Ignore opening hours, contact details, and prose.
- Normalise dietary tags to lowercase from this set only:
  vegan, vegetarian, halal, gluten-free.
  Map common codes: (V)/V -> vegetarian, (VG)/VG/Vegan -> vegan, (H)/H -> halal,
  GF -> gluten-free. Strip these codes out of the dish 'name'.
- Clean dish names: remove leading bullets, asterisks, and allergen codes.
- If a course (starter/main/side/dessert) is indicated, set it; otherwise null.
- If you cannot find a menu at all, return the college with an empty days list.
- Never invent dishes. Only report what the text supports.
"""


def extract_menu(college: str, page_text: str) -> CollegeMenu:
    """Stage 3: turn reduced page text into a validated CollegeMenu.

    Returns a CollegeMenu (Stage 4 validation happens via response_schema +
    Pydantic parsing below).
    """
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"College: {college}\n\nPage text:\n{page_text}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=CollegeMenu,
            temperature=0.0,  # deterministic-as-possible extraction
        ),
    )
    # The SDK parses JSON into our Pydantic model on .parsed when a schema is set.
    return response.parsed


print("Extractor ready.")

Extractor ready.


## 5. Stage 1-4 end to end, for one college

Run the full pipeline on a single source first. Always debug one before looping
over all — if this fails you want a clean stack trace, not four interleaved ones.

In [25]:
target = SOURCES[0]  # Queens'

raw = fetch_html(target["url"])
clean = html_to_text(raw)
menu = extract_menu(target["college"], clean)

print(f"College: {menu.college}")
print(f"Week commencing: {menu.week_commencing}")
print(f"Days parsed: {len(menu.days)}\n")

# Show the first day as a sanity check.
if menu.days:
    d = menu.days[0]
    print(f"--- {d.day} ({d.date}) ---")
    for sitting in d.meals:
        print(f"  {sitting.meal.value.upper()}")
        for dish in sitting.dishes:
            tags = f"  [{', '.join(dish.dietary)}]" if dish.dietary else ""
            print(f"    - {dish.name}{tags}")

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

## 6. Run all fetchable colleges

Now loop. We wrap each college in try/except so one broken site does not kill the
whole batch — a production scraper must be resilient to any single source failing.
JS-only sources are skipped with an explicit note.

In [ ]:
import time

results: dict[str, CollegeMenu] = {}
errors: dict[str, str] = {}

for src in SOURCES:
    college = src["college"]
    if src["mode"] == "js":
        errors[college] = "Skipped: JavaScript-rendered, needs headless browser."
        print(f"[skip] {college}: JS-rendered")
        continue
    try:
        text = html_to_text(fetch_html(src["url"]))
        results[college] = extract_menu(college, text)
        n_days = len(results[college].days)
        print(f"[ ok ] {college}: {n_days} days")
        time.sleep(7)  # stay under the 10 req/min free-tier limit
    except Exception as exc:  # noqa: BLE001 - we want to log any failure and continue
        errors[college] = repr(exc)
        print(f"[fail] {college}: {exc}")

print(f"\nSucceeded: {list(results)}")
print(f"Issues:    {errors}")

## 7. The unified output

Here is the payload your website would consume: a single JSON document holding
every college in the identical shape. This is the proof that the anti-corruption
layer worked — the consumer code below has no idea Churchill used tables and
St John's used headings.

In [ ]:
import json

unified = {
    "generated_with": MODEL_NAME,
    "colleges": [m.model_dump() for m in results.values()],
}

# Save it — this is the artifact a website backend would read.
with open("unified_menus.json", "w", encoding="utf-8") as f:
    json.dump(unified, f, indent=2, ensure_ascii=False)

print(f"Wrote unified_menus.json with {len(unified['colleges'])} colleges.\n")
print(json.dumps(unified, indent=2, ensure_ascii=False)[:1200])

## 8. A tiny consumer view

Proof that downstream code is now trivial and source-agnostic. This is the kind
of loop your website template would run. Note it touches *zero* college-specific
logic.

In [ ]:
for college_data in unified["colleges"]:
    print(f"\n{'='*50}\n {college_data['college']}  "
          f"(w/c {college_data.get('week_commencing')})\n{'='*50}")
    for day in college_data["days"][:2]:  # first two days for brevity
        print(f"\n  {day['day']}")
        for sitting in day["meals"]:
            dishes = ", ".join(d["name"] for d in sitting["dishes"][:4])
            print(f"    {sitting['meal']}: {dishes}...")

## 9. Where this goes next (notes for the script port)

When you turn this into the production `.py` for your website, the learning
checklist:

1. **Caching.** Do not call the LLM on every page load. Menus change weekly.
   Fetch + extract on a schedule (a cron job / GitHub Action once a day), write
   `unified_menus.json`, and have the website read the cached file. This also
   keeps you comfortably inside the free tier.
2. **Wolfson and other JS sites.** Swap `fetch_html` for a Playwright-based
   fetcher *only* for sources flagged `mode="js"`. The rest of the pipeline is
   unchanged — that is the payoff of keeping fetch separate.
3. **Validation as a gate.** Right now a bad extraction still gets saved. Add a
   sanity check (e.g. "did we get at least one day with at least one dish?")
   before overwriting the good cached file. Never let a bad run destroy a good
   one.
4. **Model swappability.** `MODEL_NAME` is the only model reference. To A/B
   against another provider, wrap `extract_menu` behind a small interface and
   write a second implementation. Nothing else changes.
5. **Cost / quota awareness.** Free tier is ~1,500 requests/day. At one call per
   college per day you are using a handful. You will not get close.

You now have a working anti-corruption layer. The engineering lesson: the LLM did
not replace your architecture — it became one well-isolated component inside it.
